# ADC2023 FMPE Runbook

This notebook is the single runbook for the self-contained non-quantum ADC2023 workflow in `models/sbi_ariel_adc2023`.

It covers:
- environment bootstrap
- raw dataset preflight
- prepared split generation
- train or resume launch in `tmux`
- `train.log` and periodic mRMSE monitoring
- final evaluation and prediction export

The only external dependency is the raw dataset itself, typically at `data/full-ariel`.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "models" / "sbi_ariel_adc2023").exists():
            return candidate
    raise RuntimeError("Could not find the repo root containing models/sbi_ariel_adc2023.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
WORKFLOW_DIR = PROJECT_ROOT / "models" / "sbi_ariel_adc2023"
DATA_ROOT = PROJECT_ROOT / "data" / "full-ariel"
PREPARED_DATA = PROJECT_ROOT / "data" / "generated-data" / "adc2023_fivegas_fmpe_prepared"
RUN_ROOT = PROJECT_ROOT / "local_runs"
RUN_DIR = RUN_ROOT / "sbi_ariel_adc2023_manual"
VENV_DIR = PROJECT_ROOT / ".venv-sbi-adc2023"
SETTINGS_FILE = WORKFLOW_DIR / "settings" / "adc2023_rtx4090.yaml"
LAUNCHER = WORKFLOW_DIR / "run_train_ubuntu4090.sh"
REQUIREMENTS_FILE = WORKFLOW_DIR / "requirements-vast.txt"
PYTORCH_INDEX_URL = os.environ.get("PYTORCH_INDEX_URL", "https://download.pytorch.org/whl/cu121")
SESSION_NAME = os.environ.get("ADC2023_TMUX_SESSION", "adc2023_fmpe")


def run(cmd, *, cwd: Path | None = None, env: dict[str, str] | None = None, check: bool = True) -> subprocess.CompletedProcess:
    printable = cmd if isinstance(cmd, str) else shlex.join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(
        cmd,
        cwd=str(cwd or PROJECT_ROOT),
        env=env,
        check=check,
        text=True,
        capture_output=True,
        shell=isinstance(cmd, str),
    )


PYTHON_BIN = VENV_DIR / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "workflow_dir": str(WORKFLOW_DIR),
    "data_root": str(DATA_ROOT),
    "prepared_data": str(PREPARED_DATA),
    "run_dir": str(RUN_DIR),
    "launcher": str(LAUNCHER),
}, indent=2))


## Bootstrap

Run this once per machine to create the venv and install the FMPE stack for this folder.


In [ ]:
if not VENV_DIR.exists():
    run(["python3", "-m", "venv", str(VENV_DIR)], cwd=PROJECT_ROOT)

run([str(PYTHON_BIN), "-m", "pip", "install", "--upgrade", "pip"], cwd=PROJECT_ROOT)
run([str(PYTHON_BIN), "-m", "pip", "install", "--index-url", PYTORCH_INDEX_URL, "torch"], cwd=PROJECT_ROOT)
run([str(PYTHON_BIN), "-m", "pip", "install", "--no-deps", "dingo-gw==0.8.3"], cwd=PROJECT_ROOT)
run([str(PYTHON_BIN), "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)], cwd=PROJECT_ROOT)
print(PYTHON_BIN)


## Raw Data Preflight And Preparation

This checks the required ADC2023 files and then writes the prepared train, validation, holdout, and test arrays.


In [ ]:
required_files = [
    DATA_ROOT / "TrainingData" / "AuxillaryTable.csv",
    DATA_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv",
    DATA_ROOT / "TrainingData" / "SpectralData.hdf5",
    DATA_ROOT / "TestData" / "AuxillaryTable.csv",
    DATA_ROOT / "TestData" / "SpectralData.hdf5",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing ADC2023 files:\n" + "\n".join(missing))

run([
    str(PYTHON_BIN),
    "-m",
    "models.sbi_ariel_adc2023.prepare_dataset",
    "--data-root",
    str(DATA_ROOT),
    "--output",
    str(PREPARED_DATA),
    "--overwrite",
], cwd=PROJECT_ROOT)

manifest = json.loads((PREPARED_DATA / "manifest.json").read_text())
manifest["split_counts"]


## Launch Or Resume Training

This starts the folder-local launcher inside `tmux` so the job survives terminal disconnects.

The default config already uses a small periodic RMSE subset (`512` validation rows, `32` posterior samples) so the monitor stays responsive.


In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env.update(
    {
        "DATA_ROOT": str(DATA_ROOT),
        "PREPARED_DATA": str(PREPARED_DATA),
        "RUN_DIR": str(RUN_DIR),
        "SETTINGS_FILE": str(SETTINGS_FILE),
        "VENV_DIR": str(VENV_DIR),
        "PYTORCH_INDEX_URL": PYTORCH_INDEX_URL,
    }
)

run(["tmux", "kill-session", "-t", SESSION_NAME], cwd=PROJECT_ROOT, env=env, check=False)
launch_cmd = " && ".join(
    [
        f"cd {shlex.quote(str(PROJECT_ROOT))}",
        f"export DATA_ROOT={shlex.quote(str(DATA_ROOT))}",
        f"export PREPARED_DATA={shlex.quote(str(PREPARED_DATA))}",
        f"export RUN_DIR={shlex.quote(str(RUN_DIR))}",
        f"export SETTINGS_FILE={shlex.quote(str(SETTINGS_FILE))}",
        f"export VENV_DIR={shlex.quote(str(VENV_DIR))}",
        f"export PYTORCH_INDEX_URL={shlex.quote(PYTORCH_INDEX_URL)}",
        f"bash {shlex.quote(str(LAUNCHER))}",
    ]
)
run(["tmux", "new-session", "-d", "-s", SESSION_NAME, launch_cmd], cwd=PROJECT_ROOT, env=env)
print(f"tmux attach -t {SESSION_NAME}")
print(RUN_DIR / "train.log")


## Monitor Progress

Use this to inspect the live log and the periodic RMSE records without attaching to `tmux`.


In [ ]:
log_path = RUN_DIR / "train.log"
if log_path.exists():
    tail = run(["tail", "-n", "40", str(log_path)], cwd=PROJECT_ROOT, check=False)
    print(tail.stdout)
else:
    print(f"Missing log file: {log_path}")

rmse_path = RUN_DIR / "validation_rmse.jsonl"
if rmse_path.exists():
    records = [json.loads(line) for line in rmse_path.read_text().splitlines() if line.strip()]
    for record in records[-5:]:
        print(json.dumps(record, indent=2, sort_keys=True))
else:
    print(f"No periodic RMSE file yet: {rmse_path}")


## Final Evaluation And Export

Run this after training completes or after you want to evaluate the current best checkpoint.


In [ ]:
evaluation = run([
    str(PYTHON_BIN),
    "-m",
    "models.sbi_ariel_adc2023.evaluate",
    "--run-dir",
    str(RUN_DIR),
    "--prepared-data",
    str(PREPARED_DATA),
], cwd=PROJECT_ROOT, check=False)
print(evaluation.stdout)
if evaluation.stderr:
    print(evaluation.stderr)
